# Chapter 3

## Summarizing a document bigger than the LLM’s context window

In [11]:
with open("./Moby-Dick-th.txt", 'r', encoding='utf-8') as f:
    moby_dick_book = f.read()

In [2]:
from langchain_openai import ChatOpenAI
from langchain_text_splitters import TokenTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel
import getpass

In [3]:
OPENAI_API_KEY = getpass.getpass('Enter your OPENAI_API_KEY')

Enter your OPENAI_API_KEY ········


In [4]:
llm = ChatOpenAI(openai_api_key=OPENAI_API_KEY,model_name="gpt-5-nano")

In [12]:
# Split
text_chunks_chain = (
    RunnableLambda(lambda x: 
        [
            {
                'chunk': text_chunk, 
            }
            for text_chunk in 
               TokenTextSplitter(chunk_size=3000, chunk_overlap=100).split_text(x)
        ]
    )
)

In [13]:
# Map
summarize_chunk_prompt_template = """
เขียนสรุปเนื้อหาต่อไปนี้อย่างกระชับ และรวมรายละเอียดสำคัญไว้ให้ครบถ้วน
ข้อความ: {chunk}
"""

summarize_chunk_prompt = PromptTemplate.from_template(summarize_chunk_prompt_template)
summarize_chunk_chain = summarize_chunk_prompt | llm

summarize_map_chain = (
    RunnableParallel (
        {
            'summary': summarize_chunk_chain | StrOutputParser()        
        }
    )
)

In [14]:
# Reduce
summarize_summaries_prompt_template = """
เขียนสรุปสั้นๆ จากข้อความต่อไปนี้ ซึ่งรวมเอาสรุปหลายๆ ส่วนเข้าด้วยกัน และครอบคลุมรายละเอียดหลักทั้งหมด
ข้อความ: {summaries}
"""

summarize_summaries_prompt = PromptTemplate.from_template(summarize_summaries_prompt_template)
summarize_reduce_chain = (
    RunnableLambda(lambda x: 
        {
            'summaries': '\n'.join([i['summary'] for i in x]), 
        })
    | summarize_summaries_prompt 
    | llm 
    | StrOutputParser()
)

In [15]:
map_reduce_chain = (
   text_chunks_chain
   | summarize_map_chain.map()
   | summarize_reduce_chain
)     

In [16]:
summary = map_reduce_chain.invoke(moby_dick_book)

In [17]:
print(summary)

สรุปสั้นๆ รวมรายละเอียดหลักทั้งหมด

- ข้อมูลอีบุ๊ก: PROJECT GUTENBERG เรื่อง Moby-Dick; หรือ วาฬ โดย เฮอร์แมน เมลวิลล์ จัดทำโดย แดเนียล ลาซารัส, โจนซีย์ และ เดวิด วิดเจอร์ เนื้อหาภาษาอังกฤษ ใช้รหัส UTF-8 แจกฟรีในสหรัฐอเมริกาและพื้นที่ส่วนใหญ่ของโลก ใบอนุญาตเป็นแบบ Project Gutenberg Link: www.gutenberg.org หมายเหตุทางกฎหมาย: ตรวจสอบกฎหมายท้องถิ่นหากอยู่นอกสหรัฐ

- บทนำสรุปจาก Loomings (บทที่ 1):
  - ผู้เล่า Ishmael เล่าถึงความหดหู่ทางจิตใจ มีเงินไม่มาก และไม่มีสิ่งดึงดูดบนฝั่ง เขาตัดสินใจไปทะเลเพื่อคลายความหดหู่และปรับสมดุลชีวิต
  - แนวคิดปรัชญา: ความหดหู่มักบรรเทาลงด้วยการไปทะเล เปรียบเสมือนการแทนที่ความรุนแรง/อาวุธด้วยการกระทำที่กล้าหาญ (อ้างถึงคาโต) เพื่อหาทางออกชีวิต
  - บทนี้เป็นการวางโทนเรื่อง ท่ามกลางประเด็นทะเล ความโดดเดี่ยวของมนุษย์ และการค้นหาความหมายของชีวิต
  - หมายเหตุ: บทที่ 1 ตัดคำท้ายไว้ ไม่สมบูรณ์ในข้อความที่ให้มา

- บทที่ 2 The Carpet-Bag (ตอนหลายชื่อในข้อความรวมถึงการเดินทางไป New Bedford/Nantucket):
  - Ishmael ขนเสื้อผ้าใส่กระเป๋าและออกจากแมนฮัตตันไปยัง Cape Horn/Na

## Summarizing across documents

In [23]:
from langchain_community.document_loaders import WikipediaLoader

wikipedia_loader = WikipediaLoader(query="Paestum", load_max_docs=2)
wikipedia_docs = wikipedia_loader.load()

In [29]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import TextLoader

word_loader = Docx2txtLoader("Paestum/Paestum-Britannica.docx")
word_docs = word_loader.load()

pdf_loader = PyPDFLoader("Paestum/PaestumRevisited.pdf")
pdf_docs = pdf_loader.load()

txt_loader = TextLoader("Paestum/Paestum-Encyclopedia.txt")
txt_docs = txt_loader.load()

In [30]:
all_docs = wikipedia_docs + word_docs + pdf_docs + txt_docs

In [31]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
import getpass

In [15]:
OPENAI_API_KEY = getpass.getpass('Enter your OPENAI_API_KEY')

Enter your OPENAI_API_KEY ········


In [32]:
llm = ChatOpenAI(openai_api_key=OPENAI_API_KEY,model_name="gpt-5-nano")

In [33]:
doc_summary_template = """Write a concise summary of the following text:
{text}
DOC SUMMARY:"""
doc_summary_prompt = PromptTemplate.from_template(doc_summary_template)

doc_summary_chain = doc_summary_prompt | llm

In [34]:
refine_summary_template = """
Your must produce a final summary from the current refined summary
which has been generated so far and from the content of an additional document.
This is the current refined summary generated so far: {current_refined_summary}
This is the content of the additional document: {text}
Only use the content of the additional document if it is useful, 
otherwise return the current full summary as it is."""

refine_summary_prompt = PromptTemplate.from_template(refine_summary_template)

refine_chain = refine_summary_prompt | llm | StrOutputParser()

In [35]:
def refine_summary(docs):

    intermediate_steps = []
    current_refined_summary = ''
    for doc in docs:
        intermediate_step = \
           {"current_refined_summary": current_refined_summary, 
            "text": doc.page_content}
        intermediate_steps.append(intermediate_step)
        
        current_refined_summary = refine_chain.invoke(intermediate_step)
        
    return {"final_summary": current_refined_summary,
            "intermediate_steps": intermediate_steps}

In [36]:
full_summary = refine_summary(all_docs)
print(full_summary)

{'final_summary': 'Integrated final summary (fully merged with useful content from the additional document)\n\nPaestum, originally Poseidonia, is an ancient Greek city on the Tyrrhenian coast of Magna Graecia, located in what is now Capaccio Paestum, Campania, Italy. It lies about 22 miles southeast of Salerno and near the Cilento coast.\n\nHistory\n\n- Greek foundation and early urban life: Poseidonia emerged around 600 BCE as a Greek polis, with some sources linking its founding to Sybaris. Strabo notes connections to Sybaris and cites Is of Helice as founder, reflecting a debate about Achaean vs. Sybarite leadership. The city flourished for around two centuries, enclosed by defensive walls with four gates.\n\n- Sacred and civic layout: Within the walls, three Doric temples were erected in the 6th–5th centuries BCE (Temple of Hera I, Temple of Hera II, and the Temple of Athena). An agora stood north of the Hera sanctuary, and features such as a bouleuterion or possible ekklesiasterio